# Colab 环境初始化（混合存储方案）

每次打开新的 Colab session 时，**从上到下依次运行所有 cell** 即可。

### 存储分布

| 存储位置 | 内容 | 说明 |
|---------|------|------|
| UCSD OneDrive (5 TB) | `dataset.zip`（含 `dataset/` + `pretrained_models/`） | 每次 session 下载到 Colab 本地 SSD |
| UCSD Google Drive (5 GB) | `saved_models/`, `results/` | 原生挂载，checkpoint 自动持久化 |
| GitHub (main) | `.ipynb` 代码 + `APISR_tools/` | `git clone` 拉取 |

### 前置准备（只做一次）

1. **OneDrive**: 上传 `dataset.zip`（含 dataset/ 和 pretrained_models/），获取共享直链
2. **Google Drive**: 在 `My Drive/AWM/` 下创建 `saved_models/` 和 `results/` 文件夹

In [ ]:
# ================================================
# 0. 配置
# ================================================

GITHUB_REPO = 'https://github.com/liqiqinaOH7/AWM.git'
BRANCH      = 'main'

# OneDrive 直链（原链接末尾 ?e=xxx 改成 ?download=1）
ONEDRIVE_ZIP_URL = 'https://ucsdcloud-my.sharepoint.com/:u:/g/personal/xil326_ucsd_edu/IQDFdjcJu1AAQ67p-ln0IB85AdUSjend_ov_nfx_A6DNlX0?download=1'

# Google Drive（仅存 checkpoint 和推理结果）
DRIVE_ROOT = '/content/drive/MyDrive/AWM'

# Colab 本地路径
PROJECT_DIR = '/content/AWM'

In [ ]:
# ================================================
# 1. 挂载 Google Drive
# ================================================
from google.colab import drive
drive.mount('/content/drive')

import os
for d in ['saved_models', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print(f'Drive 目录就绪: {DRIVE_ROOT}')

In [ ]:
# ================================================
# 2. 从 OneDrive 下载并解压 dataset.zip
#    zip 内含 dataset/ 和 pretrained_models/
#    每次新 session 都需要运行（Colab 本地盘是临时的）
# ================================================
import os, time

zip_path = '/content/dataset.zip'
marker = '/content/dataset/highres/original'

if os.path.isdir(marker):
    print('dataset 已存在，跳过下载。')
else:
    print('正在从 OneDrive 下载 dataset.zip（~3.4 GB）...')
    t0 = time.time()
    !wget -q --show-progress -O "{zip_path}" "{ONEDRIVE_ZIP_URL}"
    elapsed = time.time() - t0

    if os.path.isfile(zip_path) and os.path.getsize(zip_path) > 1_000_000:
        size_gb = os.path.getsize(zip_path) / 1e9
        print(f'下载完成: {size_gb:.2f} GB, 耗时 {elapsed:.0f}s')
        print('正在解压 ...')
        !unzip -q -o "{zip_path}" -d /content/
        os.remove(zip_path)
        print('解压完成，zip 已删除。')
    else:
        size = os.path.getsize(zip_path) if os.path.isfile(zip_path) else 0
        print(f'⚠ 下载可能失败（文件大小: {size} bytes）')
        print('请检查:')
        print('  1. OneDrive 链接权限是否为 "任何人可查看"')
        print('  2. 链接末尾是否为 ?download=1')
        print('  如果 wget 不行，可以尝试用下面的 cell 手动上传')

In [ ]:
# ================================================
# 2b. 备用方案：如果 OneDrive 直链下载失败
#     手动把 dataset.zip 上传到 Colab 后运行此 cell
# ================================================
# 方法 A: 从 Colab 左侧文件面板手动上传 dataset.zip 到 /content/
# 方法 B: 取消下面注释，用浏览器上传

# from google.colab import files
# uploaded = files.upload()  # 会弹出文件选择框

# import os
# zip_path = '/content/dataset.zip'
# if os.path.isfile(zip_path):
#     !unzip -q -o "{zip_path}" -d /content/
#     os.remove(zip_path)
#     print('解压完成。')

In [ ]:
# ================================================
# 3. 从 GitHub 拉取 / 更新代码
# ================================================
import os

if os.path.isdir(PROJECT_DIR):
    print('项目目录已存在，拉取最新代码 ...')
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    print('克隆仓库 ...')
    !git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'\n当前工作目录: {os.getcwd()}')

In [ ]:
# ================================================
# 4. 创建软链接
#    dataset / pretrained_models → Colab 本地（从 OneDrive 解压的）
#    saved_models / results      → Google Drive（持久化）
# ================================================
import os

links = {
    'dataset':           '/content/dataset',
    'pretrained_models': '/content/pretrained_models',
    'saved_models':      f'{DRIVE_ROOT}/saved_models',
    'results':           f'{DRIVE_ROOT}/results',
}

for name, target in links.items():
    link_path = os.path.join(PROJECT_DIR, name)
    if os.path.islink(link_path):
        existing = os.readlink(link_path)
        if existing == target:
            print(f'  OK: {name} → {target}')
            continue
        os.unlink(link_path)
    elif os.path.isdir(link_path):
        print(f'  跳过 (真实目录已存在): {name}')
        continue
    os.symlink(target, link_path)
    print(f'  链接: {name} → {target}')

print('\n软链接创建完毕。')

In [ ]:
# ================================================
# 5. 安装额外依赖
# ================================================
!pip install -q pyiqa tqdm

In [ ]:
# ================================================
# 6. 验证环境
# ================================================
import glob, os
import torch
import cv2

print('=== 系统 ===')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'OpenCV  : {cv2.__version__}')

print('\n=== 目录 ===')
for name, target in links.items():
    path = os.path.join(PROJECT_DIR, name)
    exists = os.path.isdir(path)
    if exists:
        items = os.listdir(path)
        print(f'  {name:20s} ✓ ({len(items)} items)')
    else:
        print(f'  {name:20s} ✗ 不存在')

print('\n=== 数据 ===')
hr_dir = 'dataset/highres/original'
hr_count = len(glob.glob(os.path.join(hr_dir, '*.[jp][pn]*g'))) if os.path.isdir(hr_dir) else 0
print(f'HR 图片数: {hr_count}')

for lr_name in ['lowres_2x', 'lowres_4x', 'lowres_4x_simple']:
    lr_dir = f'dataset/{lr_name}/original'
    if os.path.isdir(lr_dir):
        cnt = len(os.listdir(lr_dir))
        print(f'{lr_name:20s}: {cnt} files')

pt_files = glob.glob('pretrained_models/*.pth') if os.path.isdir('pretrained_models') else []
print(f'\n预训练权重: {[os.path.basename(f) for f in pt_files]}')

sm_files = glob.glob('saved_models/*.pth') if os.path.isdir('saved_models') else []
print(f'已训练权重: {len(sm_files)} 个 checkpoint')

print('\n✅ 环境就绪！' if hr_count > 0 else '\n⚠ 未检测到 HR 图片，请检查 dataset 下载/解压是否成功。')